cleaning crashes table


In [17]:

import pandas as pd
import sys
sys.path.append("..")
 
from config import REGION, REGION_RAW_DIR, REGION_PROCESSED_DIR 

In [18]:
print (REGION_RAW_DIR)

/workspaces/-SmartRoad-AI-for-Road-Maintenance-and-Safety/data/raw/sf


In [19]:
crashes = pd.read_csv(REGION_RAW_DIR / "crashes.csv")
streets = pd.read_csv(REGION_RAW_DIR / "streets.csv")
pavement = pd.read_csv(REGION_RAW_DIR / "pavement.csv")

print ("len crashes: ", len(crashes), "\nlen streets: ", len(streets), "\nlen pavement: ", len(pavement),)



/tmp/ipykernel_2579/427040300.py:1: DtypeWarning: Columns (3: case_id_pkey) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv(REGION_RAW_DIR / "crashes.csv")


len crashes:  66384 
len streets:  17176 
len pavement:  19461


In [20]:
cols = REGION["tables"]["crashes"]["columns"]
cols_names = list(cols.values()) # original names
rename_map = {v: k for k, v in cols.items()} # original -> unified

crashes = crashes[cols_names] # select columns
crashes = crashes.rename(columns=rename_map) # rename
print(crashes.columns.tolist())

['crash_id', 'date', 'lat', 'lon', 'intersection_id', 'intersection_type', 'killed', 'injured']


In [21]:
crashes["date"] = pd.to_datetime(crashes["date"], errors="coerce") # convert to datetime, invalid parsing will be set as NaT
crashes["year"] = crashes["date"].dt.year

print (crashes.dtypes)
print(crashes["date"].isna().sum())

crash_id                     object
date                 datetime64[us]
lat                         float64
lon                         float64
intersection_id             float64
intersection_type               str
killed                      float64
injured                       int64
year                          int32
dtype: object
0


In [22]:
crashes["year"].value_counts().sort_index() 

year
2005    3253
2006    2896
2007    3065
2008    3039
2009    2900
2010    3095
2011    3097
2012    3269
2013    3069
2014    2967
2015    3055
2016    3323
2017    3427
2018    3300
2019    3486
2020    2454
2021    2758
2022    2886
2023    2923
2024    3026
2025    2946
2026    2150
Name: count, dtype: int64

In [23]:
print("before:", len(crashes))
crashes = crashes.dropna(subset=["intersection_id"])
print("after dropna:", len(crashes))

# 2.  convert id to integer
crashes["intersection_id"] = crashes["intersection_id"].astype(int)

# 3.  check the values first
print(crashes["intersection_type"].value_counts())

# 4.  keep intersection crashes only
intersection_types = ["Intersection <= 20ft", "Intersection Rear End <= 150ft"]
crashes = crashes[crashes["intersection_type"].isin(intersection_types)].copy()
print("after filter:", len(crashes))

before: 66384
after dropna: 66318
intersection_type
Intersection <= 20ft              43543
Midblock > 20ft                   18310
Intersection Rear End <= 150ft     4464
Name: count, dtype: int64
after filter: 48007


In [24]:
crashes.to_csv(REGION_PROCESSED_DIR / "crashes_clean.csv", index=False)


Cleaning streets table

In [25]:
 #1.  keep active, non-dropped segments
mask = (streets["active"]) & (streets["date_dropped"].isna())
streets = streets[mask].copy()
print("after filter:", len(streets))

# 2. select needed columns
street_cols = ["cnn", "street", "st_type", "f_node_cnn", "t_node_cnn",
               "classcode", "oneway", "nhood"]
streets = streets[street_cols].copy()

# 3.  check missing values
print(streets["f_node_cnn"].isna().sum(), streets["t_node_cnn"].isna().sum())

# 4.  drop rows with missing nodes
streets = streets.dropna(subset=["f_node_cnn", "t_node_cnn"])

# 5. convert to integers
streets["f_node_cnn"] = streets["f_node_cnn"].astype(int)
streets["t_node_cnn"] = streets["t_node_cnn"].astype(int)

print("final:", len(streets))
print("classcode missing:", streets["classcode"].isna().sum())

# 6.  save
streets.to_csv(REGION_PROCESSED_DIR / "streets_clean.csv", index=False)


after filter: 16378
0 0
final: 16378
classcode missing: 0


cleaning pavement table

In [26]:
pavement["pci_change_date"] = pd.to_datetime(pavement["pci_change_date"], errors="coerce")
pavement = pavement.sort_values("pci_change_date", na_position="first").drop_duplicates(subset=["cnn"], keep="last")
pavemnt_cols = ["cnn", "pci_score", "pci_change_date", "treatment_or_survey", "latitude",
               "longitude"]
pavement  = pavement[pavemnt_cols].copy()
print("final:", len(pavement))
print(pavement["cnn"].is_unique)
print(pavement["pci_score"].describe())

final: 19060
True
count    19060.000000
mean        81.684942
std         17.994197
min          4.000000
25%         72.000000
50%         84.000000
75%        100.000000
max        100.000000
Name: pci_score, dtype: float64


In [27]:
pavement.to_csv(REGION_PROCESSED_DIR / "pavement_clean.csv", index=False)
